In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
TARGET = 'price_in_USD'
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Data sources for the five approaches
DATA_DIR = Path('preprocessed_data')
approach_files = {
    'approach1': DATA_DIR / 'real_estate_approach1_percentile_log_top8.csv',
    'approach2': DATA_DIR / 'real_estate_approach2_iqr_log_top8.csv',
    'approach3': DATA_DIR / 'real_estate_approach3_minimal_raw_top8.csv',
    'approach4': DATA_DIR / 'real_estate_approach4_zscore_raw_top8.csv',
    'approach5': DATA_DIR / 'real_estate_approach5_no_filtering_top8.csv'
}

def evaluate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

In [ ]:
print('Each approach trains exactly 4 models: Random Forest, HistGradientBoosting, CatBoost_Tuned, XGBoost_Optimized.')
print('XGBoost_Optimized is optimized by tuning capacity (n_estimators, max_depth), learning rate,')
print('subsampling (subsample/colsample_*), and regularization (reg_alpha/reg_lambda/gamma/min_child_weight).')

In [ ]:
def make_xgb_optimized(use_gpu: bool):
    common = dict(
        # aligned to 02_model_training_and_evaluation.ipynb (core tuned starter params)
        n_estimators=600, max_depth=8, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
        objective='reg:squarederror'
    )
    if use_gpu:
        return XGBRegressor(**common, tree_method='gpu_hist', predictor='gpu_predictor')
    return XGBRegressor(**common, tree_method='hist')

def make_catboost_tuned():
    return CatBoostRegressor(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        subsample=0.9,
        min_data_in_leaf=20,
        l2_leaf_reg=3.0,
        random_state=RANDOM_STATE,
        verbose=False
    )

def make_models(use_gpu_for_xgb: bool):
    return {
        'Random Forest': RandomForestRegressor(
            n_estimators=300, max_depth=18, min_samples_split=4, min_samples_leaf=2,
            max_features=0.6, n_jobs=-1, random_state=RANDOM_STATE
        ),
        'HistGradientBoosting': HistGradientBoostingRegressor(
            max_iter=350, learning_rate=0.05, max_depth=12,
            min_samples_leaf=15, l2_regularization=0.8, random_state=RANDOM_STATE
        ),
        'CatBoost_Tuned': make_catboost_tuned(),
        'XGBoost_Optimized': make_xgb_optimized(use_gpu_for_xgb),
    }

def build_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', ohe)
    ])
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    transformers = []
    if cat_cols:
        transformers.append(('cat', cat_pipe, cat_cols))
    if num_cols:
        transformers.append(('num', num_pipe, num_cols))
    return ColumnTransformer(transformers)

def run_approach(approach_name, csv_path):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.125, random_state=RANDOM_STATE
    )
    models = make_models(use_gpu_for_xgb=True)
    rows = []
    for model_name, model in tqdm(list(models.items()), desc=f'{approach_name}: models', leave=False):
        pipe = Pipeline([('prep', build_preprocessor(X_train)), ('model', model)])
        used_device = None
        if model_name == 'XGBoost_Optimized':
            try:
                pipe.fit(X_train, y_train)
                used_device = 'gpu'
            except Exception:
                pipe = Pipeline([('prep', build_preprocessor(X_train)), ('model', make_xgb_optimized(use_gpu=False))])
                pipe.fit(X_train, y_train)
                used_device = 'cpu'
        else:
            pipe.fit(X_train, y_train)
        val_metrics = evaluate_metrics(y_val, pipe.predict(X_val))
        test_metrics = evaluate_metrics(y_test, pipe.predict(X_test))
        rows.append({
            'approach': approach_name,
            'model': model_name,
            'device': used_device if used_device is not None else 'cpu',
            'val_mae': val_metrics['mae'],
            'val_rmse': val_metrics['rmse'],
            'val_r2': val_metrics['r2'],
            'test_mae': test_metrics['mae'],
            'test_rmse': test_metrics['rmse'],
            'test_r2': test_metrics['r2']
        })
    return pd.DataFrame(rows)

results = []
for name, path in tqdm(list(approach_files.items()), desc='Approaches'):
    results.append(run_approach(name, path))
results_df = pd.concat(results, ignore_index=True)
# Choose models using validation RMSE; test metrics are reporting-only.
results_df_sorted = results_df.sort_values(['approach', 'val_rmse']).reset_index(drop=True)
best_per_approach = results_df_sorted.groupby('approach', as_index=False).head(1).reset_index(drop=True)
results_df[['approach','model','device']].drop_duplicates().sort_values(['approach','model'])
results_df_sorted

In [ ]:
# Phase 1: compare the 4 models inside each approach (validation RMSE)
g = sns.catplot(
    data=results_df_sorted,
    x='model', y='val_rmse',
    col='approach', col_wrap=3,
    kind='bar', height=3.4, aspect=1.2
)
g.set_titles('{col_name}')
g.set_xticklabels(rotation=20)
g.fig.suptitle('Phase 1: validation RMSE by model within each approach', y=1.04)
plt.tight_layout()
plt.show()
best_per_approach.sort_values('val_rmse')

In [ ]:
# Phase 2: compare the best model from each approach
plt.figure(figsize=(8, 4))
order = best_per_approach.sort_values('val_rmse')['approach']
ax = sns.barplot(data=best_per_approach, x='approach', y='val_rmse', order=order)
ax.set_title('Phase 2: validation-selected model per approach')
ax.set_xlabel('approach')
ax.set_ylabel('validation RMSE')
ax.tick_params(axis='x', rotation=20)
for i, r in best_per_approach.set_index('approach').loc[list(order)].reset_index().iterrows():
    ax.text(i, r['val_rmse'], str(r['model']), ha='center', va='bottom', fontsize=8, rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Compact overview heatmap (validation RMSE)
pivot_rmse = results_df.pivot(index='approach', columns='model', values='val_rmse')
plt.figure(figsize=(8, 3.6))
sns.heatmap(pivot_rmse, annot=True, fmt='.0f', cmap='viridis')
plt.title('Validation RMSE heatmap (approach x model)')
plt.tight_layout()
plt.show()

In [ ]:
# Final table for reporting
best_per_approach.sort_values('val_rmse').reset_index(drop=True)